Name: Yash Chafekanade  
Roll No: A-77  
Class: TY AI & DS  
Subject: ANNDL  
Assignment: CIE2  


Title:
Customer Segmentation Analysis using Boltzmann Machine
Based on Online Retail Shopping Habits

Objective:
To develop a Boltzmann Machine model to categorize
customers based on their shopping habits.

In [4]:
# CIA-2: Customer Segmentation using Boltzmann Machine

# Task 1: Import Libraries

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

# Task 2: Load Dataset

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

data = pd.read_excel(url)

print("Original Data Shape:", data.shape)

# Task 3: Clean Data

data = data.dropna()

print("After Cleaning Shape:", data.shape)

print(data.head())

# Task 4: Create Customer-Item Matrix

pivot_table = pd.pivot_table(
    data,
    values='Quantity',
    index='CustomerID',
    columns='StockCode',
    aggfunc='sum',
    fill_value=0
)

print("Pivot Table Shape:", pivot_table.shape)

# Task 5: Preprocess Data (Scaling)

# Convert all column names to strings to resolve TypeError with MinMaxScaler
pivot_table.columns = pivot_table.columns.astype(str)

scaler = MinMaxScaler()

pivot_scaled = scaler.fit_transform(pivot_table)

print("Scaled Data Sample:")
print(pivot_scaled[:5])

# Task 6: Convert to Binary

pivot_scaled[pivot_scaled > 0] = 1

training_set = torch.FloatTensor(pivot_scaled)

print("Binary Data Sample:")
print(training_set[:5])

# Task 7: Define RBM

class RBM():

    def __init__(self, nv, nh):

        self.W = torch.randn(nh, nv)

        self.a = torch.randn(1, nh)

        self.b = torch.randn(1, nv)

    def sample_h(self, x):

        wx = torch.mm(x, self.W.t())

        activation = wx + self.a.expand_as(wx)

        p_h_given_v = torch.sigmoid(activation)

        return p_h_given_v, torch.bernoulli(p_h_given_v)

    def sample_v(self, y):

        wy = torch.mm(y, self.W)

        activation = wy + self.b.expand_as(wy)

        p_v_given_h = torch.sigmoid(activation)

        return p_v_given_h, torch.bernoulli(p_v_given_h)

    def train(self, v0, vk, ph0, phk):

        self.W += torch.mm(ph0.t(), v0) - torch.mm(phk.t(), vk)

        self.b += torch.sum((v0 - vk), 0)

        self.a += torch.sum((ph0 - phk), 0)

# Task 8: Train Boltzmann Machine

nv = len(training_set[0])

nh = 100

rbm = RBM(nv, nh)

nb_epoch = 5

for epoch in range(nb_epoch):

    loss = 0

    for id_user in range(len(training_set)):

        v0 = training_set[id_user:id_user+1]

        vk = training_set[id_user:id_user+1]

        ph0,_ = rbm.sample_h(v0)

        for k in range(5):

            _,hk = rbm.sample_h(vk)

            _,vk = rbm.sample_v(hk)

        phk,_ = rbm.sample_h(vk)

        rbm.train(v0, vk, ph0, phk)

        loss += torch.mean(torch.abs(v0 - vk))

    print("Epoch:", epoch+1,
          "Loss:", loss.item()/len(training_set))

print("Training Completed!")

Original Data Shape: (541909, 8)
After Cleaning Shape: (406829, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
Pivot Table Shape: (4372, 3684)
Scaled Data Sample:
[[0.         0.         0.         ... 0.23340249 0.         0.0535714